# 02 — Perfil preliminar de la cohorte adulta

Construye un perfil agregado de las estancias UCI del demo local. La edad se reconstruye mediante `anchor_age + año(intime) - anchor_year`. Este paso es descriptivo: todavía no aplica la definición de sepsis ni constituye la cohorte analítica final.

**Entrada:** MIMIC-IV Demo v2.2 descargado por el notebook 00. **Salida:** tablas agregadas sin registros individuales.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data' / 'mimic-iv-demo' / '2.2'
required_files = [
    DATA_DIR / 'hosp' / 'patients.csv.gz',
    DATA_DIR / 'hosp' / 'admissions.csv.gz',
    DATA_DIR / 'icu' / 'icustays.csv.gz',
]
missing = [path for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(f'Faltan archivos; ejecute primero el notebook 00: {missing}')
connection = duckdb.connect()

In [ ]:
patients_path, admissions_path, icustays_path = map(str, required_files)
connection.read_csv(patients_path).create_view('patients')
connection.read_csv(admissions_path).create_view('admissions')
connection.read_csv(icustays_path).create_view('icustays')
connection.execute(
    '''
    CREATE OR REPLACE TEMP VIEW cohort_stays AS
    SELECT
        i.subject_id, i.hadm_id, i.stay_id,
        i.first_careunit, i.last_careunit, i.intime, i.outtime, i.los,
        a.admission_type, a.admission_location, a.discharge_location,
        a.hospital_expire_flag, p.gender,
        p.anchor_age + year(i.intime) - p.anchor_year AS age_at_icu
    FROM icustays AS i
    JOIN patients AS p USING (subject_id)
    JOIN admissions AS a USING (subject_id, hadm_id)
    '''
)
print('Vista temporal cohort_stays creada.')

## Flujo de inclusión preliminar

In [ ]:
cohort_flow = connection.execute(
    '''
    SELECT
      count(*) AS linked_icu_stays,
      count(*) FILTER (WHERE age_at_icu >= 18) AS adult_icu_stays,
      count(*) FILTER (WHERE age_at_icu >= 18 AND outtime > intime) AS valid_adult_stays,
      count(DISTINCT subject_id) FILTER (WHERE age_at_icu >= 18 AND outtime > intime) AS adult_patients,
      count(DISTINCT hadm_id) FILTER (WHERE age_at_icu >= 18 AND outtime > intime) AS adult_admissions
    FROM cohort_stays
    '''
).df()
cohort_flow

## Resumen agregado de estancias adultas válidas

In [ ]:
adult_summary = connection.execute(
    '''
    SELECT
      min(age_at_icu) AS min_age,
      median(age_at_icu) AS median_age,
      max(age_at_icu) AS max_age,
      round(median(los), 2) AS median_icu_los_days,
      round(quantile_cont(los, 0.25), 2) AS q1_icu_los_days,
      round(quantile_cont(los, 0.75), 2) AS q3_icu_los_days,
      sum(hospital_expire_flag) AS hospital_deaths
    FROM cohort_stays
    WHERE age_at_icu >= 18 AND outtime > intime
    '''
).df()
adult_summary

In [ ]:
careunit_summary = connection.execute(
    '''
    SELECT first_careunit, count(*) AS icu_stays,
           count(DISTINCT subject_id) AS patients
    FROM cohort_stays
    WHERE age_at_icu >= 18 AND outtime > intime
    GROUP BY first_careunit
    ORDER BY icu_stays DESC, first_careunit
    '''
).df()
careunit_summary

## Reingresos y unidad de análisis

In [ ]:
repeat_stays = connection.execute(
    '''
    WITH per_admission AS (
      SELECT subject_id, hadm_id, count(*) AS stays_per_admission
      FROM cohort_stays
      WHERE age_at_icu >= 18 AND outtime > intime
      GROUP BY subject_id, hadm_id
    )
    SELECT stays_per_admission, count(*) AS admissions
    FROM per_admission
    GROUP BY stays_per_admission
    ORDER BY stays_per_admission
    '''
).df()
repeat_stays

## Interpretación y siguiente paso

El demo permite comprobar relaciones y código, pero sus 100 pacientes no representan una muestra de desarrollo. Antes del notebook 03 debemos congelar si la cohorte primaria conservará la primera estancia UCI por ingreso o todas las estancias, manteniendo siempre las particiones agrupadas por `subject_id`.

In [ ]:
connection.close()